# ⚡ Score Maximiser — Plain CNN Edition

| Tier | Correct | Reward |
|------|---------|--------|
| 1 | ≤ 5,000 | 100 € each |
| 2 | 5,001 – 6,000 | 200 € each |
| 3 | > 6,000 | 1,000 € each |

**Net Score = Reward − Total Parameters**

Architecture: **3-block CNN with BatchNorm + residual shortcuts**  
- Three convolutional blocks, each doubling the filters  
- BatchNorm after every conv for stable, fast training  
- Residual (skip) connections prevent vanishing gradients  
- Global Average Pooling instead of Flatten — kills millions of params  
- Result: high accuracy with a very small parameter count

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import ssl; ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import warnings; warnings.filterwarnings('ignore')

print(f'TF {tf.__version__}  |  GPU: {len(tf.config.list_physical_devices("GPU"))} device(s)')

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────────────────────
CFG = {
    # Dataset: 'fashion_mnist' | 'mnist' | 'cifar10' | 'cifar100'
    'dataset'    : 'cifar10',

    # CNN architecture
    'filters'    : [32, 64, 128],   # filters per block (3 blocks)
    'dropout'    : [0.1, 0.2, 0.3], # dropout after each block
    'dense_units': 256,             # units in the dense head
    'dense_drop' : 0.4,             # dropout before the output layer

    # Training
    'epochs'     : 40,
    'batch_size' : 128,
    'lr'         : 1e-3,
    'weight_decay': 1e-4,

    'save_path'  : 'cnn_model.keras',
}
EVAL_SIZE = 10_000

In [ ]:
# ── Cell 3: Load & Preprocess ─────────────────────────────────────────────────
LOADERS = {
    'fashion_mnist': keras.datasets.fashion_mnist,
    'mnist':         keras.datasets.mnist,
    'cifar10':       keras.datasets.cifar10,
    'cifar100':      keras.datasets.cifar100,
}
assert CFG['dataset'] in LOADERS, f"Unknown dataset. Choose from {list(LOADERS)}"

(x_train, y_train), (x_test, y_test) = LOADERS[CFG['dataset']].load_data()

# Normalise to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# Grayscale (H, W) → (32, 32, 3)  so the same model handles every dataset
if x_train.ndim == 3:
    x_train = x_train[..., np.newaxis]
    x_test  = x_test[..., np.newaxis]
if x_train.shape[-1] == 1:
    x_train = np.concatenate([x_train]*3, axis=-1)
    x_test  = np.concatenate([x_test] *3, axis=-1)
if x_train.shape[1] != 32:
    x_train = tf.image.resize(x_train, [32, 32]).numpy()
    x_test  = tf.image.resize(x_test,  [32, 32]).numpy()

y_train = y_train.flatten()
y_test  = y_test.flatten()
NUM_CLASSES = int(y_train.max()) + 1

# Manual val split — required when using tf.data.Dataset
n_val  = int(len(x_train) * 0.1)
x_val, y_val = x_train[:n_val], y_train[:n_val]
x_tr,  y_tr  = x_train[n_val:], y_train[n_val:]
x_eval, y_eval = x_test[:EVAL_SIZE], y_test[:EVAL_SIZE]

print(f"dataset : {CFG['dataset']}  |  classes : {NUM_CLASSES}")
print(f"train   : {len(x_tr):,}  |  val : {len(x_val):,}  |  eval : {len(x_eval):,}")
print(f"shape   : {x_tr.shape[1:]}")

In [ ]:
# ── Cell 4: Augmentation + tf.data pipelines ──────────────────────────────────
aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1),
], name='augmentation')

ds_train = (
    tf.data.Dataset.from_tensor_slices((x_tr, y_tr))
    .shuffle(50_000)
    .batch(CFG['batch_size'])
    .map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)
ds_val = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(CFG['batch_size'])
    .prefetch(tf.data.AUTOTUNE)
)
print(f"train batches: {len(ds_train)}  |  val batches: {len(ds_val)}")

In [ ]:
# ── Cell 5: Build CNN ─────────────────────────────────────────────────────────
#
# Each block follows:  Conv → BN → ReLU → Conv → BN → ReLU → MaxPool → Dropout
# A 1×1 residual shortcut matches the channel count so the skip connection works.
# Global Average Pooling collapses the spatial dims to a single vector per filter,
# avoiding the massive Flatten → Dense param explosion.

def conv_block(x, filters, dropout_rate, name):
    shortcut = x

    x = layers.Conv2D(filters, 3, padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4),
                      name=f'{name}_c1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.Activation('relu', name=f'{name}_act1')(x)

    x = layers.Conv2D(filters, 3, padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(1e-4),
                      name=f'{name}_c2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)

    # Residual shortcut: 1×1 conv to match filter count if needed
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, use_bias=False,
                                  name=f'{name}_skip')(shortcut)
        shortcut = layers.BatchNormalization(name=f'{name}_skip_bn')(shortcut)

    x = layers.Add(name=f'{name}_add')([x, shortcut])
    x = layers.Activation('relu', name=f'{name}_act2')(x)
    x = layers.MaxPooling2D(name=f'{name}_pool')(x)
    x = layers.Dropout(dropout_rate, name=f'{name}_drop')(x)
    return x


def build_cnn(input_shape, num_classes, filters, dropout, dense_units, dense_drop):
    inp = keras.Input(shape=input_shape, name='input')

    # Stem: lightweight first conv
    x = layers.Conv2D(filters[0], 3, padding='same', use_bias=False,
                      name='stem_conv')(inp)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.Activation('relu', name='stem_act')(x)

    # Three residual blocks
    for i, (f, d) in enumerate(zip(filters, dropout)):
        x = conv_block(x, f, d, name=f'block{i+1}')

    # Head: GAP → Dense → Softmax
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(dense_units, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     name='dense')(x)
    x = layers.Dropout(dense_drop, name='head_drop')(x)
    out = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return keras.Model(inp, out, name='PlainCNN')


model = build_cnn(
    input_shape = x_tr.shape[1:],
    num_classes = NUM_CLASSES,
    filters     = CFG['filters'],
    dropout     = CFG['dropout'],
    dense_units = CFG['dense_units'],
    dense_drop  = CFG['dense_drop'],
)

model.summary()
print(f"\nTotal parameters  : {model.count_params():,}")
print(f"Parameter penalty : -{model.count_params():,}")

In [ ]:
# ── Cell 6: Model diagram ─────────────────────────────────────────────────────
try:
    import pydot  # noqa
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pydot', '-q'])

try:
    keras.utils.plot_model(
        model, to_file='model_diagram.png',
        show_shapes=True, show_layer_names=True,
        show_dtype=False, dpi=96
    )
    from IPython.display import Image, display
    display(Image('model_diagram.png'))
    print('Diagram saved → model_diagram.png')
except Exception as e:
    print('[Diagram skipped] Install graphviz: brew install graphviz')
    print(f'Reason: {e}')
    model.summary()

In [ ]:
# ── Cell 7: Compile ───────────────────────────────────────────────────────────
# Plain float lr so ReduceLROnPlateau can adjust it.
# (LearningRateSchedule objects are read-only after creation and crash callbacks.)

model.compile(
    optimizer = keras.optimizers.AdamW(
        learning_rate = CFG['lr'],
        weight_decay  = CFG['weight_decay'],
    ),
    loss    = 'sparse_categorical_crossentropy',
    metrics = ['accuracy'],
)
print('Compiled.')

In [ ]:
# ── Cell 8: Train ─────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint(CFG['save_path'], monitor='val_accuracy',
                    save_best_only=True, verbose=0),
]

history = model.fit(
    ds_train,
    validation_data = ds_val,
    epochs          = CFG['epochs'],
    callbacks       = callbacks,
    verbose         = 1,
)

print(f"\nBest val accuracy : {max(history.history['val_accuracy']):.4f}")
print(f"Model saved → {CFG['save_path']}")

In [ ]:
# ── Cell 9: Training curves ───────────────────────────────────────────────────
hist = history.history
epcs = range(1, len(hist['loss']) + 1)

fig = plt.figure(figsize=(14, 4))
gs  = gridspec.GridSpec(1, 3, figure=fig)

ax1 = fig.add_subplot(gs[0])
ax1.plot(epcs, hist['loss'],     label='Train', linewidth=2)
ax1.plot(epcs, hist['val_loss'], label='Val',   linewidth=2, linestyle='--')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(gs[1])
ax2.plot(epcs, hist['accuracy'],     label='Train', linewidth=2)
ax2.plot(epcs, hist['val_accuracy'], label='Val',   linewidth=2, linestyle='--')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(gs[2])
gap = np.array(hist['accuracy']) - np.array(hist['val_accuracy'])
ax3.fill_between(epcs, gap, alpha=0.4, color='tomato')
ax3.plot(epcs, gap, color='tomato', linewidth=2)
ax3.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax3.set_title('Overfit gap (train − val acc)')
ax3.set_xlabel('Epoch'); ax3.grid(alpha=0.3)

plt.suptitle('Training Diagnostics', fontsize=12)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=96)
plt.show()
print('Saved → training_curves.png')

In [ ]:
# ── Cell 10: Evaluate & Score ─────────────────────────────────────────────────
y_pred_probs = model.predict(x_eval, batch_size=256, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
correct      = int(np.sum(y_pred == y_eval))
total_params = model.count_params()

tier1 = min(correct, 5_000) * 100
tier2 = max(0, min(correct, 6_000) - 5_000) * 200
tier3 = max(0, correct - 6_000) * 1_000
reward    = tier1 + tier2 + tier3
net_score = reward - total_params

print()
print('╔══════════════════════════════════════════╗')
print('║          SCORE REPORT                    ║')
print('╠══════════════════════════════════════════╣')
print(f'║  Correct predictions : {correct:>7,} / {EVAL_SIZE:,}   ║')
print(f'║  Accuracy            : {correct/EVAL_SIZE:>10.2%}         ║')
print('╠══════════════════════════════════════════╣')
print(f'║  Tier 1 (≤5k  × 100€)  : {tier1:>12,} €  ║')
print(f'║  Tier 2 (5k–6k× 200€)  : {tier2:>12,} €  ║')
print(f'║  Tier 3 (>6k  ×1000€)  : {tier3:>12,} €  ║')
print(f'║  Gross reward          : {reward:>12,} €  ║')
print(f'║  Parameter penalty     : {total_params:>12,}    ║')
print('╠══════════════════════════════════════════╣')
print(f'║  NET SCORE             : {net_score:>12,}    ║')
print('╚══════════════════════════════════════════╝')

In [ ]:
# ── Cell 11: Confusion matrix ─────────────────────────────────────────────────
CLASS_NAMES = {
    'fashion_mnist': ['T-shirt','Trouser','Pullover','Dress','Coat',
                      'Sandal','Shirt','Sneaker','Bag','Ankle boot'],
    'mnist':         list('0123456789'),
    'cifar10':       ['plane','car','bird','cat','deer',
                      'dog','frog','horse','ship','truck'],
    'cifar100':      [str(i) for i in range(100)],
}
names = CLASS_NAMES.get(CFG['dataset'], [str(i) for i in range(NUM_CLASSES)])

if NUM_CLASSES <= 20:
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for t, p in zip(y_eval, y_pred):
        cm[t, p] += 1

    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            if cm[i, j] > 0:
                color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
                ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                        fontsize=7, color=color)
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=96)
    plt.show()
    print('Saved → confusion_matrix.png')

In [ ]:
# ── Cell 12: Sample predictions with confidence bars ─────────────────────────
rng  = np.random.default_rng(42)
idxs = rng.choice(EVAL_SIZE, size=8, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(18, 5),
                          gridspec_kw={'height_ratios': [3, 1]})

for col, i in enumerate(idxs):
    img   = x_eval[i]
    probs = y_pred_probs[i]
    pred  = int(np.argmax(probs))
    true  = int(y_eval[i])
    ok    = pred == true

    ax_img = axes[0, col]
    show   = img.squeeze() if img.shape[-1] == 1 else img
    ax_img.imshow(show, cmap='gray' if show.ndim == 2 else None)
    ax_img.set_title(
        f"{'✓' if ok else '✗'} {names[pred]}\n({names[true]})",
        color='green' if ok else 'red', fontsize=8
    )
    ax_img.axis('off')

    ax_bar = axes[1, col]
    colors = ['steelblue'] * NUM_CLASSES
    colors[pred] = 'green' if ok else 'red'
    ax_bar.bar(range(NUM_CLASSES), probs, color=colors, width=0.8)
    ax_bar.set_ylim(0, 1)
    ax_bar.set_xticks([]); ax_bar.set_yticks([])
    ax_bar.set_xlabel('confidence', fontsize=7)

plt.suptitle('Predictions  (✓ correct  ✗ wrong) — bar = class confidence', fontsize=10)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=96)
plt.show()
print('Saved → sample_predictions.png')

In [ ]:
# ── Cell 13: Reward curve ─────────────────────────────────────────────────────
x_axis = np.arange(0, 10_001)
reward_curve = np.where(
    x_axis <= 5_000, x_axis * 100,
    np.where(x_axis <= 6_000,
             5_000*100 + (x_axis - 5_000)*200,
             5_000*100 + 1_000*200 + (x_axis - 6_000)*1_000)
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_axis, reward_curve / 1_000, color='steelblue', linewidth=2,
        label='Gross reward (k€)')
ax.axvline(correct, color='tomato', linewidth=2, linestyle='--',
           label=f'Your model ({correct:,} correct)')
ax.axvline(5_000, color='gray', linewidth=1, linestyle=':', label='Tier boundary 5k')
ax.axvline(6_000, color='gray', linewidth=1, linestyle=':', label='Tier boundary 6k')
ax.fill_betweenx([0, reward_curve.max()/1_000], 0, correct, alpha=0.08, color='tomato')
ax.set_xlabel('Correct predictions out of 10,000')
ax.set_ylabel('Gross reward (k€)')
ax.set_title('Reward curve — where your model lands')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=96)
plt.show()
print('Saved → reward_curve.png')

## Tuning guide

| Change | Effect |
|--------|--------|
| `'filters': [64, 128, 256]` | Richer features → higher accuracy, more params |
| `'filters': [16, 32, 64]` | Fewer params, lower penalty, modest accuracy drop |
| `'dense_units': 128` | Smaller head → fewer params, small accuracy cost |
| Remove dense layer entirely (set to 0 and skip) | GAP output feeds directly to softmax — ultra-lean |
| Increase `'dropout'` values | Less overfitting on small datasets |
| `'dataset': 'fashion_mnist'` | Easier than CIFAR-10, easier to push into Tier 3 |

**Sweet spot:** aim for 8,000+ correct (Tier 3 = 1,000 € each) while keeping params under ~300,000.